# 3D WTMM Analysis — Python Port of xsmurf

Complete implementation of the 3D Wavelet Transform Modulus Maxima (WTMM) method,
ported from the xsmurf C codebase. Extends the 2D pipeline to volumetric data.

1. **FFT-based 3D CWT** with Gaussian derivative wavelets (ψ_x, ψ_y, ψ_z)
2. **Gradient modulus** (scalar 3D field)
3. **Non-maxima suppression** with trilinear interpolation (from `edge/extrema_core.c`)
4. **Tensor SVD** for 3D vector fields (3×3 Jacobian)
5. **Chain/surface extraction** via 3D connected component labeling
6. **Partition function** Z(q,a) and **D(h) singularity spectrum**
7. **Visualization** with 3D scatter plots

In [ ]:
import numpy as np
from numpy.fft import fftn, ifftn, fftfreq
from scipy.ndimage import label as ndimage_label
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import warnings
warnings.filterwarnings('ignore')

print(f"NumPy {np.__version__}")
print("Backend: NumPy FFT (CPU)")

## 1. Test Volume — 3D Fractional Brownian Field

Generate a 3D fBm with known Hurst exponent H for validation.

In [ ]:
def generate_fbm_3d(N, H=0.5, seed=42):
    """Generate 3D fractional Brownian field with Hurst exponent H.
    
    Power spectrum P(k) ~ |k|^{-(2H+3)} in 3D.
    """
    rng = np.random.default_rng(seed)
    
    kx = fftfreq(N, d=1.0/N)
    ky = fftfreq(N, d=1.0/N)
    kz = fftfreq(N, d=1.0/N)
    KX, KY, KZ = np.meshgrid(kx, ky, kz, indexing='ij')
    K = np.sqrt(KX**2 + KY**2 + KZ**2)
    K[0, 0, 0] = 1.0
    
    # 3D: amplitude exponent = -(2H+3)/2 = -(H+1.5)
    amplitude = K ** (-(H + 1.5))
    amplitude[0, 0, 0] = 0.0
    
    phases = rng.uniform(0, 2*np.pi, (N, N, N))
    spectrum = amplitude * np.exp(1j * phases)
    field = np.real(ifftn(spectrum))
    
    field = (field - field.min()) / (field.max() - field.min())
    return field

# Use smaller size for 3D to keep computation tractable
N3 = 64
H_true = 0.7

print(f"Generating {N3}x{N3}x{N3} fBm volume (H={H_true})...")
volume = generate_fbm_3d(N3, H=H_true)
print(f"Volume shape: {volume.shape}")
print(f"Value range: [{volume.min():.4f}, {volume.max():.4f}]")

# Visualize central slices
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
mid = N3 // 2
axes[0].imshow(volume[mid, :, :], cmap='gray')
axes[0].set_title(f'XY slice (z={mid})')
axes[1].imshow(volume[:, mid, :], cmap='gray')
axes[1].set_title(f'XZ slice (y={mid})')
axes[2].imshow(volume[:, :, mid], cmap='gray')
axes[2].set_title(f'YZ slice (x={mid})')
plt.suptitle(f'3D fBm Volume (H={H_true}, N={N3})', fontsize=14)
plt.tight_layout()
plt.show()

## 2. 3D Wavelet Definitions

Extension of the 2D Gaussian derivative wavelets to 3D:
- ψ_x(kx, ky, kz) = i·kx · exp(-(kx² + ky² + kz²))
- ψ_y(kx, ky, kz) = i·ky · exp(-(kx² + ky² + kz²))
- ψ_z(kx, ky, kz) = i·kz · exp(-(kx² + ky² + kz²))

In [ ]:
def gaussian_derivative_wavelet_3d_fourier(kx_grid, ky_grid, kz_grid, scale):
    """
    3D Gaussian derivative wavelets in Fourier domain.
    
    Extension of xsmurf's __conv_dx, __conv_dy to 3D.
    """
    akx = scale * kx_grid
    aky = scale * ky_grid
    akz = scale * kz_grid
    
    gauss = np.exp(-(akx**2 + aky**2 + akz**2))
    
    psi_x = 1j * akx * gauss
    psi_y = 1j * aky * gauss
    psi_z = 1j * akz * gauss
    
    return psi_x, psi_y, psi_z

print("3D wavelet definitions loaded.")

## 3. FFT-based 3D CWT

In [ ]:
def compute_scales_3d(n_octaves, n_voices, a_min=1.0):
    """Generate scale sequence (same as 2D)."""
    scales = []
    norm_factor = 6.0 / 0.86
    for oct in range(n_octaves):
        for vox in range(n_voices):
            s = a_min * (2.0 ** (oct + vox / n_voices)) * norm_factor
            scales.append(s)
    return np.array(scales)

def cwt_3d(volume, scales):
    """
    3D Continuous Wavelet Transform using FFT convolution.
    
    Returns:
        wx, wy, wz: arrays (n_scales, nz, ny, nx)
    """
    nz, ny, nx = volume.shape
    n_scales = len(scales)
    
    kx = fftfreq(nx) * 2 * np.pi
    ky = fftfreq(ny) * 2 * np.pi
    kz = fftfreq(nz) * 2 * np.pi
    KX, KY, KZ = np.meshgrid(kx, ky, kz, indexing='ij')
    
    vol_fft = fftn(volume)
    
    wx = np.zeros((n_scales, nz, ny, nx), dtype=np.float64)
    wy = np.zeros((n_scales, nz, ny, nx), dtype=np.float64)
    wz = np.zeros((n_scales, nz, ny, nx), dtype=np.float64)
    
    for i, scale in enumerate(scales):
        psi_x, psi_y, psi_z = gaussian_derivative_wavelet_3d_fourier(KX, KY, KZ, scale)
        wx[i] = np.real(ifftn(vol_fft * np.conj(psi_x)))
        wy[i] = np.real(ifftn(vol_fft * np.conj(psi_y)))
        wz[i] = np.real(ifftn(vol_fft * np.conj(psi_z)))
    
    return wx, wy, wz

# Compute 3D CWT (fewer scales for tractability)
n_oct_3d = 3
n_vox_3d = 2
scales_3d = compute_scales_3d(n_oct_3d, n_vox_3d, a_min=1.0)

print(f"Computing 3D CWT with {len(scales_3d)} scales...")
print(f"Scale range: [{scales_3d[0]:.2f}, {scales_3d[-1]:.2f}]")

wx3, wy3, wz3 = cwt_3d(volume, scales_3d)
print(f"CWT shape: wx={wx3.shape}")
print("Done.")

## 4. 3D Gradient Modulus (from `edge/misc.c`)

Port of `GradientModulus3D`: modulus = sqrt(wx² + wy² + wz²)

In [ ]:
def gradient_modulus_3d(wx, wy, wz):
    """Port of xsmurf GradientModulus3D (edge/misc.c)."""
    return np.sqrt(wx**2 + wy**2 + wz**2)

# Compute modulus at all scales
modulus_3d = gradient_modulus_3d(wx3, wy3, wz3)
print(f"3D Modulus shape: {modulus_3d.shape}")

# Visualize central slices at different scales
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
mid = N3 // 2
for col, idx in enumerate([0, len(scales_3d)//2, len(scales_3d)-1]):
    axes[0, col].imshow(modulus_3d[idx, mid, :, :], cmap='hot')
    axes[0, col].set_title(f'XY slice (a={scales_3d[idx]:.1f})')
    axes[0, col].axis('off')
    axes[1, col].imshow(modulus_3d[idx, :, mid, :], cmap='hot')
    axes[1, col].set_title(f'XZ slice (a={scales_3d[idx]:.1f})')
    axes[1, col].axis('off')
plt.suptitle('3D Wavelet Transform Modulus', fontsize=14)
plt.tight_layout()
plt.show()

## 5. 3D Non-Maxima Suppression (from `edge/extrema_core.c`)

Port of `Remove_Gradient_NonMaxima_Slice_3D`. Uses trilinear interpolation
to check if each voxel is a local maximum along the 3D gradient direction.

xsmurf processes 3D data slice-by-slice, with 3-buffer scheme (prev/current/next).
Here we operate on the full volume for simplicity.

In [ ]:
def non_maxima_suppression_3d(gx, gy, gz, norme, eps_norm=5e-7, eps_deriv=0.9995):
    """
    Port of xsmurf Remove_Gradient_NonMaxima_Slice_3D (edge/extrema_core.c).
    
    Non-maxima suppression along 3D gradient direction using trilinear interpolation.
    
    The xsmurf version processes one z-slice at a time using 3 norme buffers.
    This version operates on the full volume.
    """
    nz, ny, nx = norme.shape
    maxima = np.zeros_like(norme)
    
    for z in range(1, nz - 1):
        for y in range(1, ny - 1):
            for x in range(1, nx - 1):
                n = norme[z, y, x]
                
                if n < eps_norm:
                    continue
                
                # Normalize gradient
                ngx = gx[z, y, x] / n
                ngy = gy[z, y, x] / n
                ngz = gz[z, y, x] / n
                
                # Nearest neighbor case
                if (abs(ngx) > eps_deriv or abs(ngy) > eps_deriv or abs(ngz) > eps_deriv):
                    xi = int(x + ngx + 0.5)
                    yi = int(y + ngy + 0.5)
                    zi = int(z + ngz + 0.5)
                    if 0 <= xi < nx and 0 <= yi < ny and 0 <= zi < nz:
                        if n <= norme[zi, yi, xi]:
                            continue
                    xi = int(x - ngx + 0.5)
                    yi = int(y - ngy + 0.5)
                    zi = int(z - ngz + 0.5)
                    if 0 <= xi < nx and 0 <= yi < ny and 0 <= zi < nz:
                        if n < norme[zi, yi, xi]:
                            continue
                    maxima[z, y, x] = n
                    continue
                
                # Trilinear interpolation
                # Forward point
                xf = x + ngx
                yf = y + ngy
                zf = z + ngz
                
                if (xf < 0 or xf >= nx-1 or yf < 0 or yf >= ny-1 or
                    zf < 0 or zf >= nz-1):
                    continue
                
                x0 = int(xf)
                y0 = int(yf)
                z0 = int(zf)
                dx = xf - x0
                dy = yf - y0
                dz = zf - z0
                
                # Trilinear coefficients (from xsmurf)
                dxdy = dx * dy
                dydz = dy * dz
                dxdz = dx * dz
                c111 = dxdy * dz
                c011 = dydz - c111
                c101 = dxdz - c111
                c001 = dz - dxdz - c011
                c110 = dxdy - c111
                c010 = dy - dxdy - c011
                c100 = dx - dxdy - c101
                c000 = 1.0 - dx - dy + dxdy - c001
                
                interp = (norme[z0, y0, x0] * c000 +
                         norme[z0, y0, x0+1] * c100 +
                         norme[z0, y0+1, x0] * c010 +
                         norme[z0, y0+1, x0+1] * c110 +
                         norme[z0+1, y0, x0] * c001 +
                         norme[z0+1, y0, x0+1] * c101 +
                         norme[z0+1, y0+1, x0] * c011 +
                         norme[z0+1, y0+1, x0+1] * c111)
                
                if n <= interp:
                    continue
                
                # Backward point — reuse coefficients (swapped)
                xb = x - ngx
                yb = y - ngy
                zb = z - ngz
                
                if (xb < 0 or xb >= nx-1 or yb < 0 or yb >= ny-1 or
                    zb < 0 or zb >= nz-1):
                    continue
                
                x0b = int(xb)
                y0b = int(yb)
                z0b = int(zb)
                
                interp_b = (norme[z0b, y0b, x0b] * c111 +
                           norme[z0b, y0b, x0b+1] * c011 +
                           norme[z0b, y0b+1, x0b] * c101 +
                           norme[z0b, y0b+1, x0b+1] * c001 +
                           norme[z0b+1, y0b, x0b] * c110 +
                           norme[z0b+1, y0b, x0b+1] * c010 +
                           norme[z0b+1, y0b+1, x0b] * c100 +
                           norme[z0b+1, y0b+1, x0b+1] * c000)
                
                if n < interp_b:
                    continue
                
                maxima[z, y, x] = n
    
    return maxima

# Compute 3D extrema
print("Computing 3D non-maxima suppression (this may take a while)...")
extrema_3d = np.zeros_like(modulus_3d)
for i in range(len(scales_3d)):
    print(f"  Scale {i} (a={scales_3d[i]:.1f})...", end=' ')
    extrema_3d[i] = non_maxima_suppression_3d(wx3[i], wy3[i], wz3[i], modulus_3d[i])
    n_ext = np.count_nonzero(extrema_3d[i])
    print(f"{n_ext} extrema")

print(f"Total 3D extrema: {np.count_nonzero(extrema_3d)}")

In [ ]:
# Visualize 3D extrema as scatter plot
fig = plt.figure(figsize=(18, 6))

for col, idx in enumerate([0, len(scales_3d)//2, len(scales_3d)-1]):
    ax = fig.add_subplot(1, 3, col+1, projection='3d')
    
    # Get extrema positions
    ez, ey, ex = np.where(extrema_3d[idx] > 0)
    
    # Subsample if too many points
    max_pts = 2000
    if len(ez) > max_pts:
        rng = np.random.default_rng(42)
        subset = rng.choice(len(ez), max_pts, replace=False)
        ez, ey, ex = ez[subset], ey[subset], ex[subset]
    
    colors = extrema_3d[idx, ez, ey, ex]
    ax.scatter(ex, ey, ez, c=colors, cmap='hot', s=1, alpha=0.5)
    ax.set_title(f'3D Extrema (a={scales_3d[idx]:.1f}, n={np.count_nonzero(extrema_3d[idx])})')
    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    ax.set_zlabel('Z')

plt.suptitle('3D WTMM Extrema', fontsize=14)
plt.tight_layout()
plt.show()

## 6. 3D Chain Extraction and Skeleton

In 3D, the extrema form 2D surfaces (sheets) at each scale,
rather than 1D lines as in 2D. We use 3D connected component labeling
(26-connectivity) to identify these surfaces.

In [ ]:
class Extremum3D:
    """3D extremum point."""
    __slots__ = ['x', 'y', 'z', 'mod', 'scale_idx', 'component_id']
    def __init__(self, x, y, z, mod, scale_idx):
        self.x = x
        self.y = y
        self.z = z
        self.mod = mod
        self.scale_idx = scale_idx
        self.component_id = -1

class ExtVolume:
    """All extrema and connected components at one scale."""
    def __init__(self, scale_idx, scale):
        self.scale_idx = scale_idx
        self.scale = scale
        self.extrema = []
        self.components = {}  # comp_id -> list of Extremum3D
        self.n_components = 0

def extract_3d_components(extrema_map, scale_idx, scale):
    """
    Extract connected components of 3D extrema.
    Uses 26-connectivity (full 3x3x3 neighborhood).
    """
    ext_vol = ExtVolume(scale_idx, scale)
    
    binary = (extrema_map > 0).astype(np.int32)
    structure = np.ones((3, 3, 3), dtype=np.int32)  # 26-connectivity
    labeled, n_comp = ndimage_label(binary, structure=structure)
    ext_vol.n_components = n_comp
    
    positions = np.argwhere(extrema_map > 0)
    for z, y, x in positions:
        ext = Extremum3D(x, y, z, extrema_map[z, y, x], scale_idx)
        comp_id = labeled[z, y, x]
        ext.component_id = comp_id
        ext_vol.extrema.append(ext)
        
        if comp_id not in ext_vol.components:
            ext_vol.components[comp_id] = []
        ext_vol.components[comp_id].append(ext)
    
    return ext_vol

# Extract 3D components
print("Extracting 3D connected components...")
ext_volumes = []
for i in range(len(scales_3d)):
    ev = extract_3d_components(extrema_3d[i], i, scales_3d[i])
    ext_volumes.append(ev)
    print(f"  Scale {i} (a={scales_3d[i]:.1f}): {ev.n_components} components, "
          f"{len(ev.extrema)} extrema")

print(f"Total components: {sum(ev.n_components for ev in ext_volumes)}")

## 7. 3D Tensor SVD (from `edge/misc.c`)

Port of `GradientModulus3D_tensor3D` — SVD of 3×3 Jacobian tensor.
And `GradientModulus3D_tensor3D_LT` — longitudinal/transverse decomposition.

In [ ]:
SVD_TYPE_MAX = 0
SVD_TYPE_MIN = 1

def gradient_modulus_tensor_3d_vectorized(gx1, gy1, gz1,
                                          gx2, gy2, gz2,
                                          gx3, gy3, gz3,
                                          svd_type=SVD_TYPE_MAX,
                                          is_trace=False):
    """
    Port of xsmurf GradientModulus3D_tensor3D (edge/misc.c).
    
    SVD of 3x3 Jacobian tensor at each voxel:
        T = [[gx1, gx2, gx3],
             [gy1, gy2, gy3],
             [gz1, gz2, gz3]]
    
    Parameters:
        gx1..gz3: nine components of the 3x3 tensor
        svd_type: SVD_TYPE_MAX or SVD_TYPE_MIN
        is_trace: if True, modulus = sum of all singular values (trace)
    """
    shape = gx1.shape
    n = np.prod(shape)
    
    # Stack into (n, 3, 3)
    T = np.stack([
        np.stack([gx1.ravel(), gx2.ravel(), gx3.ravel()], axis=-1),
        np.stack([gy1.ravel(), gy2.ravel(), gy3.ravel()], axis=-1),
        np.stack([gz1.ravel(), gz2.ravel(), gz3.ravel()], axis=-1)
    ], axis=-2)
    
    U, s, Vt = np.linalg.svd(T)
    
    if is_trace:
        modulus = np.sum(s, axis=-1).reshape(shape)
    elif svd_type == SVD_TYPE_MAX:
        modulus = s[:, 0].reshape(shape)
    else:
        modulus = s[:, 2].reshape(shape)  # smallest
    
    return modulus

def gradient_modulus_tensor_3d_LT(gx1, gy1, gz1,
                                   gx2, gy2, gz2,
                                   gx3, gy3, gz3):
    """
    Port of xsmurf GradientModulus3D_tensor3D_LT (edge/misc.c).
    
    Longitudinal (symmetric) and transverse (antisymmetric) decomposition.
    
    modL = MAX6(|gx1|, |gy2|, |gz3|,
                0.5|gx2+gy1|, 0.5|gx3+gz1|, 0.5|gy3+gz2|)
    modT = MAX3(0.5|gx2-gy1|, 0.5|gx3-gz1|, 0.5|gy3-gz2|)
    """
    modL = np.maximum.reduce([
        np.abs(gx1), np.abs(gy2), np.abs(gz3),
        0.5 * np.abs(gx2 + gy1),
        0.5 * np.abs(gx3 + gz1),
        0.5 * np.abs(gy3 + gz2)
    ])
    
    modT = np.maximum.reduce([
        0.5 * np.abs(gx2 - gy1),
        0.5 * np.abs(gx3 - gz1),
        0.5 * np.abs(gy3 - gz2)
    ])
    
    return modL, modT

print("3D tensor functions defined.")
print("  - gradient_modulus_tensor_3d_vectorized: SVD of 3x3 Jacobian")
print("  - gradient_modulus_tensor_3d_LT: Longitudinal/transverse decomposition")

## 8. Partition Function and D(h) Spectrum (3D)

In [ ]:
def partition_function_3d(ext_volumes, scales, q_values, min_comp_size=5):
    """
    Compute WTMM partition function for 3D data.
    
    For each scale, sum |mod|^q over the sup of modulus in each component.
    """
    n_q = len(q_values)
    n_scales = len(scales)
    Z = np.zeros((n_q, n_scales))
    
    for s_idx, ext_vol in enumerate(ext_volumes):
        comp_maxima = []
        for comp_id, ext_list in ext_vol.components.items():
            if len(ext_list) >= min_comp_size:
                max_mod = max(e.mod for e in ext_list)
                if max_mod > 0:
                    comp_maxima.append(max_mod)
        
        if len(comp_maxima) == 0:
            Z[:, s_idx] = np.nan
            continue
        
        comp_maxima = np.array(comp_maxima)
        for q_idx, q in enumerate(q_values):
            if q == 0:
                Z[q_idx, s_idx] = len(comp_maxima)
            else:
                Z[q_idx, s_idx] = np.sum(comp_maxima ** q)
    
    return Z

def compute_tau_3d(Z, scales, q_values):
    """Compute τ(q) from partition function."""
    n_q = len(q_values)
    tau = np.zeros(n_q)
    log_a = np.log(scales)
    
    for q_idx in range(n_q):
        z_vals = Z[q_idx]
        valid = np.isfinite(z_vals) & (z_vals > 0)
        if np.sum(valid) < 2:
            tau[q_idx] = np.nan
            continue
        coeffs = np.polyfit(log_a[valid], np.log(z_vals[valid]), 1)
        tau[q_idx] = coeffs[0]
    
    return tau

def compute_dh_spectrum_3d(tau, q_values, d=3):
    """D(h) via Legendre transform (d=3 for 3D)."""
    h = np.gradient(tau, q_values)
    D = q_values * h - tau + d
    return h, D

# Compute partition function
q_values = np.linspace(-3, 3, 15)

print("Computing 3D partition function...")
Z3 = partition_function_3d(ext_volumes, scales_3d, q_values)
tau3 = compute_tau_3d(Z3, scales_3d, q_values)
h3, D3 = compute_dh_spectrum_3d(tau3, q_values, d=3)

print(f"τ(1) = {tau3[np.argmin(np.abs(q_values-1))]:.3f}")
print(f"τ(2) = {tau3[np.argmin(np.abs(q_values-2))]:.3f}")

In [ ]:
# Comprehensive 3D visualization
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

# 1. Partition function
ax = axes[0, 0]
for q in [-2, 0, 1, 2]:
    q_idx = np.argmin(np.abs(q_values - q))
    z_vals = Z3[q_idx]
    valid = np.isfinite(z_vals) & (z_vals > 0)
    if np.any(valid):
        ax.loglog(scales_3d[valid], z_vals[valid], 'o-', label=f'q={q}', markersize=4)
ax.set_xlabel('Scale a')
ax.set_ylabel('Z(q, a)')
ax.set_title('3D Partition Function')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# 2. τ(q)
ax = axes[0, 1]
valid_tau = np.isfinite(tau3)
ax.plot(q_values[valid_tau], tau3[valid_tau], 'b.-')
ax.plot(q_values, q_values * H_true - 3, 'r--', alpha=0.5, label=f'qH-3 (H={H_true})')
ax.set_xlabel('q')
ax.set_ylabel('τ(q)')
ax.set_title('3D Scaling Exponents')
ax.legend()
ax.grid(True, alpha=0.3)

# 3. D(h)
ax = axes[0, 2]
valid_dh = np.isfinite(h3) & np.isfinite(D3)
ax.plot(h3[valid_dh], D3[valid_dh], 'g.-', markersize=8)
ax.set_xlabel('h')
ax.set_ylabel('D(h)')
ax.set_title('3D Singularity Spectrum')
ax.axhline(y=3, color='gray', linestyle=':', alpha=0.5, label='D=3')
ax.axvline(x=H_true, color='r', linestyle='--', alpha=0.5, label=f'H={H_true}')
ax.legend()
ax.grid(True, alpha=0.3)

# 4-6. Extrema slices at different scales
mid = N3 // 2
for col, idx in enumerate([0, len(scales_3d)//2, len(scales_3d)-1]):
    ax = axes[1, col]
    ext_slice = extrema_3d[idx, mid, :, :]
    ax.imshow(ext_slice > 0, cmap='gray_r')
    n_ext = np.count_nonzero(extrema_3d[idx])
    ax.set_title(f'Extrema z={mid} (a={scales_3d[idx]:.1f}, total={n_ext})')
    ax.axis('off')

plt.suptitle(f'3D WTMM Analysis (fBm H={H_true}, N={N3})', fontsize=16)
plt.tight_layout()
plt.show()

## 9. Summary

| Component | xsmurf Source | Python Port |
|-----------|--------------|-------------|
| 3D Wavelet | Extension of `wt.tcl` | `gaussian_derivative_wavelet_3d_fourier()` |
| 3D FFT convolution | `cv2d` extended | `cwt_3d()` via numpy.fft.fftn |
| 3D Gradient modulus | `edge/misc.c` (GradientModulus3D) | `gradient_modulus_3d()` |
| 3D Non-maxima suppression | `edge/extrema_core.c` (Slice_3D) | `non_maxima_suppression_3d()` |
| 3D Component extraction | Extension of `wt2d/chain.c` | `extract_3d_components()` |
| 3D Tensor SVD | `edge/misc.c` (tensor3D) | `gradient_modulus_tensor_3d_vectorized()` |
| 3D LT decomposition | `edge/misc.c` (tensor3D_LT) | `gradient_modulus_tensor_3d_LT()` |

In [ ]:
print("="*60)
print("3D WTMM Analysis Complete")
print("="*60)
print(f"Volume size: {N3}x{N3}x{N3}")
print(f"Scales: {len(scales_3d)} ({n_oct_3d} octaves × {n_vox_3d} voices)")
print(f"Total extrema: {sum(len(ev.extrema) for ev in ext_volumes)}")
print(f"Total components: {sum(ev.n_components for ev in ext_volumes)}")